## Check generated fakes

In [1]:
import os
from PIL import Image

fake_files = sorted(f for f in os.listdir("data/fakes") if f.endswith(".png"))
print(len(fake_files), fake_files[:3], fake_files[-3:])
print({Image.open(f"data/fakes/{f}").size for f in fake_files})

2800 ['seed0000.png', 'seed0001.png', 'seed0002.png'] ['seed2797.png', 'seed2798.png', 'seed2799.png']
{(512, 512)}


## Download real images

In [2]:
import time
import requests
import pandas as pd

df = pd.read_csv("data/deepfake.csv")
real = df[df["label"] == "REAL"].copy()
real["photo_id"] = real["image_url"].str.extract(r"(photo-[0-9a-f-]+)")[0]
print(len(real), "rows,", real["photo_id"].nunique(), "unique photos")
real = real.drop_duplicates("photo_id")

2790 rows, 2349 unique photos


In [ ]:
os.makedirs("data/reals", exist_ok=True)
failures = []
session = requests.Session()

for i, row in enumerate(real.itertuples(), start=1):
    path = f"data/reals/{row.photo_id}.jpg"
    if os.path.exists(path):
        continue
    try:
        resp = session.get(row.image_url, timeout=30)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")
        if not content_type.startswith("image/"):
            raise ValueError(f"not an image: {content_type}")
        with open(path, "wb") as f:
            f.write(resp.content)
    except Exception as e:
        failures.append({"photo_id": row.photo_id, "url": row.image_url, "error": str(e)})
    if i % 100 == 0:
        print(f"{i}/{len(real)} done, {len(failures)} failures")
    time.sleep(0.2)

pd.DataFrame(failures).to_csv("data/real_download_failures.csv", index=False)
print("Finished with", len(failures), "failures")

## Try the failed downloads again

In [4]:
os.makedirs("data/reals", exist_ok=True)
failures = []
session = requests.Session()

for i, row in enumerate(real.itertuples(), start=1):
    path = f"data/reals/{row.photo_id}.jpg"
    if os.path.exists(path):
        continue
    try:
        resp = session.get(row.image_url, timeout=30)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")
        if not content_type.startswith("image/"):
            raise ValueError(f"not an image: {content_type}")
        with open(path, "wb") as f:
            f.write(resp.content)
    except Exception as e:
        failures.append({"photo_id": row.photo_id, "url": row.image_url, "error": str(e)})
    if i % 100 == 0:
        print(f"{i}/{len(real)} done, {len(failures)} failures")
    time.sleep(0.2)

pd.DataFrame(failures).to_csv("data/real_download_failures.csv", index=False)
print("Finished with", len(failures), "failures")

Finished with 0 failures
